In [1]:
#apply a previously trained model to a new dataset


# This is the project directory where everything is saved - possible to get this automatically?)
basedir = '/Users/paddyslator/python/ED/ED_MRI/examples/paper_experiments/ADC_model'

trained_model_filename = "results/ADC_model_SNR20_n_train_vox_100_all_trained_model.pt"

results_filename = "results/ADC_model_SNR20_n_train_vox_100_all.npy"

simulations_filenames = "ADC_model_SNR20_simulation_data.npy"

#basedir = "/home/blumberg/Bureau/z_Automated_Measurement/Output/tst/"



In [2]:
import torch
from omegaconf import OmegaConf
from tadred.trainer import Trainer
from tadred.data_processing import create_data_norm
from pathlib import Path


checkpoint = torch.load(
    Path(basedir,trained_model_filename),
    map_location="cpu",
    weights_only=False,
)

args = OmegaConf.create(checkpoint["args"])
data_features_norm = checkpoint["data_features_norm"]

nnet = Trainer(
    tadred_train_eval=args.tadred_train_eval,
    network=args.network,
    data_features_norm=data_features_norm,
    train_pytorch=args.train_pytorch,
    other_options=args.other_options,
)

nnet.device = "cpu"
nnet._create_model()
nnet.model.load_state_dict(checkpoint["model_state_dict"])
nnet.model.eval()




TADREDNet(
  (score_net): FCN(
    (layers): Sequential(
      (0): Linear(in_features=192, out_features=1000, bias=True)
      (1): ReLU()
      (2): Linear(in_features=1000, out_features=1000, bias=True)
      (3): ReLU()
      (4): Linear(in_features=1000, out_features=192, bias=True)
      (5): Identity()
    )
  )
  (task_net): FCN(
    (layers): Sequential(
      (0): Linear(in_features=192, out_features=1000, bias=True)
      (1): ReLU()
      (2): Linear(in_features=1000, out_features=1000, bias=True)
      (3): ReLU()
      (4): Linear(in_features=1000, out_features=1, bias=True)
      (5): Identity()
    )
  )
  (score_activation): Sigmoid()
  (downsampling_mult_layer): DownsamplingMultLayer()
  (loss_fct): MSELoss()
)

In [3]:
#load the datasets
import numpy as np
dataset = np.load(Path(basedir,simulations_filenames),allow_pickle=True).item()

In [12]:
#apply the network to the dataset
test_output_nn = nnet.model.forward_eval(dataset['test'], score=1)

print(test_output_nn)


tensor([[1.6882],
        [0.9356],
        [1.1347],
        [2.5372],
        [0.2924],
        [0.2086],
        [2.1260],
        [0.9463],
        [1.6814],
        [0.6970]], grad_fn=<MulBackward0>)


In [ ]:
#load original tadred results
results = np.load(Path(basedir,results_filename),allow_pickle=True)

In [10]:
results[12]['test_output']

array([[1.6881697 ],
       [0.935566  ],
       [1.1346632 ],
       [2.5372353 ],
       [0.29244432],
       [0.20864695],
       [2.1260083 ],
       [0.9463167 ],
       [1.6814388 ],
       [0.6970413 ]], dtype=float32)